# 03 - Evaluation

Evaluate the trained model and create final metrics/visualizations.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.append(str(PROJECT_ROOT / 'src'))

import matplotlib.pyplot as plt
import tensorflow as tf

import numpy as np
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay

from data_processing import build_image_datasets, optimize_dataset
from utils import CLASS_NAMES

In [ ]:
raw_dir = PROJECT_ROOT / 'data' / 'raw'
model_path = PROJECT_ROOT / 'models' / 'trained' / 'kidmood_mobilenetv2.keras'
results_dir = PROJECT_ROOT / 'results'
viz_dir = results_dir / 'visualizations'
viz_dir.mkdir(parents=True, exist_ok=True)

if not model_path.exists():
    print('Missing trained model:', model_path)
else:
    _, _, test_ds = build_image_datasets(raw_dir=raw_dir, batch_size=32)
    y_test = np.concatenate([labels.numpy() for _, labels in test_ds], axis=0)
    test_ds = optimize_dataset(test_ds)

    model = tf.keras.models.load_model(model_path)
    probs = model.predict(test_ds)
    y_pred = probs.argmax(axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(y_test, y_pred, average='macro')
    report = classification_report(y_test, y_pred, target_names=CLASS_NAMES)

    print('Accuracy:', accuracy)
    print('Macro F1:', macro_f1)
    print(report)

In [ ]:
if model_path.exists():
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(cmap='Blues', xticks_rotation=45)
    plt.title('KidMood Confusion Matrix')
    plt.tight_layout()
    plt.savefig(viz_dir / 'confusion_matrix.png', dpi=160)
    plt.show()